# Module 12.2 — Embedding Layer 🧠

## Part 1 — Learning

### 1. What is an embedding layer?

An embedding layer converts a token ID (an integer representing a word or token) into a dense numerical vector.

Suppose our vocabulary is:

| Word    | Token ID |
| ------- | -------- |
| `<PAD>` | 0        |
| `<UNK>` | 1        |
| `good`  | 2        |
| `movie` | 3        |
| `bad`   | 4        |

A neural network cannot extract much semantic information from the integer `2` alone. The ID is just an index, not a measurement of how good a word is.

An embedding layer uses that ID to retrieve a vector representing the token.

Input token ID

# 2

Represents "good"

Embedding lookup

Retrieve row 2 from the embedding matrix

Dense vector

## [0.8, 0.3, −0.2, 0.6]

Illustrative 4-dimensional vector

The vector above is just an example. Actual values depend on how the embeddings were initialized and trained.

### 2. The embedding matrix

An embedding layer stores a matrix of vectors, with one row for each token in the vocabulary.

Let:

- \\(V\\) = vocabulary size
- \\(D\\) = embedding dimension

Then the embedding matrix is

\\[ E\in\mathbb{R}^{V\times D} \\]

For our example, assume \\(V=5\\) and \\(D=4\\).

Example embedding matrix

| Token ID | Token   | Embedding vector          |
| -------- | ------- | ------------------------- |
| 0        | `<PAD>` | \\([0,0,0,0]\\)           |
| 1        | `<UNK>` | \\([0.1,-0.2,0.3,0.1]\\)  |
| 2        | `good`  | \\([0.8,0.3,-0.2,0.6]\\)  |
| 3        | `movie` | \\([0.2,0.5,0.1,-0.4]\\)  |
| 4        | `bad`   | \\([-0.5,0.2,0.7,-0.1]\\) |

These are illustrative values, not pretrained GloVe vectors.

The embedding layer retrieves the appropriate row using the token ID. It does not need to search through the vocabulary word by word.

Important: The integer token ID is used as an index into the matrix. It is not multiplied by the matrix as an ordinary numerical feature.



### 3. What happens when a sentence enters the embedding layer?

Suppose the input sentence is:

`"good movie good"`

Its token IDs are:

\\[ [2,3,2] \\]

The embedding layer retrieves three rows:

\\[ X = \begin{bmatrix} E_2\\\ E_3\\\ E_2 \end{bmatrix} \\]

If each embedding has four dimensions, the output shape is

\\[ \boxed{(3,4)} \\]

There are three tokens, each represented by four numbers.

Notice that `good` appears twice. Both occurrences receive the same vector from a static embedding layer. The embedding lookup itself does not understand the surrounding context.

For a batch of sentences, the output typically has shape

\\[ (B,L,D) \\]

where \\(B\\) is batch size, \\(L\\) is sequence length, and \\(D\\) is embedding dimension.

For example, with 32 sentences, 20 token positions per sentence, and 100-dimensional embeddings, the output shape is `(32, 20, 100)`.

### 4. Where do the embedding values come from?

There are two important ways to initialize the embedding matrix.

A. Random initialization

The embedding matrix begins with randomly initialized values. During model training, these values can be updated through backpropagation.

B. Pretrained initialization

We initialize the embedding matrix with pretrained vectors, such as the GloVe vectors we loaded in Module 12.1.

- Frozen: Keep the pretrained vectors fixed.
- Trainable: Allow them to adapt to the task during training.

This is why Module 12.1 and Module 12.2 are connected: Module 12.1 prepares the vectors, while Module 12.2 uses them in a PyTorch embedding layer.

### 5. Does the embedding layer learn meaning?

The layer stores vectors; the training process determines how those vectors are learned or adapted.

For randomly initialized embeddings, the model learns useful representations through training. With pretrained embeddings, it starts with representations learned from another corpus.

The individual vector dimensions generally do not have simple, directly interpretable meanings like “positive sentiment” or “noun.” Their usefulness comes from the representation as a whole and how the model learns to use it.

### What you should remember

1. An embedding layer is a lookup table mapping token IDs to dense vectors.
2. Its weight matrix has shape \\(V\times D\\).
3. A sequence of \\(L\\) tokens produces \\(L\times D\\) embeddings.
4. Token IDs are indices, not numerical measurements of word meaning.
5. A standard static embedding lookup returns the same vector for the same token ID, regardless of context.
6. Embedding weights can be randomly initialized, loaded from pretrained vectors, frozen, or fine-tuned.

---

In [1]:
# create a vocublary and tokens 

import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
    "good": 2,
    "movie": 3,
    "bad": 4
}

tokens = ["good", "movie", "good"]

token_ids = torch.tensor(
    [vocab.get(token, vocab["<UNK>"]) for token in tokens],
    dtype=torch.long,
    device=device
)

print("Token IDs:", token_ids)
print("Shape:", token_ids.shape)

Device: cuda
Token IDs: tensor([2, 3, 2], device='cuda:0')
Shape: torch.Size([3])


In [2]:
# create embedding layer 

vocab_size = len(vocab)
embedding_dim = 4

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=vocab["<PAD>"]
).to(device)

print("Embedding matrix shape:", embedding.weight.shape)

Embedding matrix shape: torch.Size([5, 4])


In [3]:
# Perform the embedding lookup

vectors = embedding(token_ids)

print("Token IDs:", token_ids)
print("Embedding vectors:\n", vectors)
print("Output shape:", vectors.shape)

Token IDs: tensor([2, 3, 2], device='cuda:0')
Embedding vectors:
 tensor([[ 1.1816,  1.2636, -0.6303,  0.5177],
        [ 1.1744,  0.6545,  0.0081,  1.9510],
        [ 1.1816,  1.2636, -0.6303,  0.5177]], device='cuda:0',
       grad_fn=<EmbeddingBackward0>)
Output shape: torch.Size([3, 4])


In [4]:
# manual check
good_vector = embedding.weight[vocab["good"]]

print("Lookup result:")
print(vectors[0])

print("\nDirect matrix row:")
print(good_vector)

print("\nAre they equal?")
print(torch.equal(vectors[0], good_vector))

Lookup result:
tensor([ 1.1816,  1.2636, -0.6303,  0.5177], device='cuda:0',
       grad_fn=<SelectBackward0>)

Direct matrix row:
tensor([ 1.1816,  1.2636, -0.6303,  0.5177], device='cuda:0',
       grad_fn=<SelectBackward0>)

Are they equal?
True


In [5]:
# Trainable embeddings
trainable_embedding = nn.Embedding(
    len(vocab),
    embedding_dim,
    padding_idx=vocab["<PAD>"]
).to(device)

# Frozen embeddings
frozen_embedding = nn.Embedding(
    len(vocab),
    embedding_dim,
    padding_idx=vocab["<PAD>"]
).to(device)

frozen_embedding.weight.requires_grad_(False)

print("Trainable:", trainable_embedding.weight.requires_grad)
print("Frozen:", frozen_embedding.weight.requires_grad)

Trainable: True
Frozen: False


In [7]:
# Understand batch dimensions

batch_ids = torch.tensor(
    [
        [2, 3, 4],
        [3, 2, 0]
    ],
    dtype=torch.long,
    device=device
)

batch_vectors = embedding(batch_ids)

print("Input shape:", batch_ids.shape)
print("Output shape:", batch_vectors.shape)

# Tensor	       Shape	  Meaning
# Input IDs	       (2, 3)	  2 sequences, 3 token positions each
# Output vectors   (2, 3, 4)  2 sequences, 3 positions, 4 dimensions per token

Input shape: torch.Size([2, 3])
Output shape: torch.Size([2, 3, 4])
